## **SAVE A COPY OF THIS NOTEBOOK TO PUT ANSWERS INTO**

#Part 1 (Reading Assignment), 20 points

**Reminder: there are 5 reading assignments, 3% each for 15% total of your final grade.**

The reading assignment consists of two papers - [one on alignment before fusion](https://arxiv.org/abs/2107.07651), and one on the [Platonic Representation Hypothesis](https://arxiv.org/pdf/2405.07987). Read both papers and then answer the following questions:

1. Explain the implications of 'align before fuse' on your tasks of interest. at which degree, and what type of fusion needs to be performed? And what level of alignment would you need to perform for your data, so that subsequent fusion or representation learning is successful?
2. How can you perform controlled experiments to validate the types of fusion and/or alignment needed for your tasks? What are some challenges you foresee in fusing and aligning your data?
3. Explain the implications of 'platonic representation hypothesis' on your tasks of interest. Do you believe alignment between modalities would automatically emerge as models trained on your data are scaled up?
4. What are some reasons why alignment would not emerge i.e., counter-arguments to the platonic representation hypothesis? You are encouraged to search for follow-up works to the original paper that both support and counter the original arguments.
5. What experiments would you propose to validate the existence and emergence of alignment in your tasks?
6. Can you also think of some downsides of strongly (or perfectly) aligning your data modalities? How can you design experiments to validate that these risks are not present in your trained models?

## Response:
1. Our dataset is an EPIC medical dataset with tabular features and clinical notes. Unlike vision–language settings where image–text pairs are natural positives, we cannot easily create positive pairs between long free-text notes and high-dimensional tabular vectors. The ALBEF paper argues that when modalities are unaligned, the fusion encoder struggles to learn cross-modal interactions. They use a contrastive loss to align image and text representations before fusing them via cross-modal attention. For our setting, we can still apply “align before fuse” by mapping both modalities into a shared embedding space (e.g., notes via BioClinicalBERT, tabular via an MLP) and encouraging alignment of patient-level representations (i.e. same note and tabular vector for a patient should be close in the shared embedding space). Given that notes and tabular features share overlapping clinical information, late fusion is a reasonable baseline (separate encoders, then combine predictions or concatenate embeddings before a head). 

2. We can validate fusion and alignment with fixed train/validation/test splits. (1) Baselines: Encode notes with BioClinicalBERT and compare late fusion (train separate predictors per modality, then combine predictions) vs. early fusion (concatenate note embeddings and tabular features, then one prediction model). (2) Alignment ablations: Add an alignment objective (e.g., contrastive or correlation loss on paired note–tabular embeddings for the same patient) before fusion. Then apply simple concatenation fusion and train the prediction head. Compare validation performance and robustness. (3) If alignment helps, try more expressive fusion (e.g., cross-attention, gated fusion). Challenges include aligning variable-length, noisy, poorly structured notes with fixed-size, binary/numeric tabular features. 

3. The hypothesis states that representations from different models and modalities converge toward a shared statistical model of reality as scale (data, model size, task diversity) increases. For our EPIC task, if this held, we would expect that as we scale up note encoders (e.g., larger LMs) and tabular models, their embedding spaces would become more aligned without explicit alignment training. 

I believe alignment would automatically emerge only partially. The Platonic paper shows convergence across vision and language on broad, internet-scale data; medical data are domain-specific, often smaller, and notes vs. tabular capture different “views” of the same underlying reality (the patient state). So some emergence seems plausible with scale, but we should not rely on it without measuring. Therefore, there is the need for explicit alignment experiments and alignment metrics (e.g., mutual nearest-neighbor) as in the paper.

4. In [“Revisiting the Platonic Representation Hypothesis: An Aristotelian View”](https://arxiv.org/pdf/2602.14486), Groger et al. argue that alignment metrics can be confounded by model scale—larger models may score higher on similarity purely due to dimensionality and aggregation, not true convergence. They propose a null-calibration framework for representational similarity, which shows that global convergence may largely disappear, with only local neighborhood structure agreeing. [Kapoor et al.](https://arxiv.org/pdf/2502.18710) also form a counter-argument against the Platonic Representation Hypothesis by observing that alignment does not hold once the data distribution changes. Furthermore, the platonic representation hypothesis might not hold depending on domain and objectives. Medical notes and tabular data are generated by different processes and may not share the same “reality” in the same way as image–caption pairs. Therefore, task and data constraints in our setting may not push representations toward the same platonic ideal.

5. First, to validate the existence of alignment in our tasks, we can measure it by usuing the mutual nearest-neighbor metric. We can also train tabular and note encoders at different capacities and check whether alignment increases with scale. Lastly, we can check to see if there are correlations between mutual nearest-neighbor values and the corresponding validation performance of the model. The Platonic representation paper suggests that higher alignment should correlate with better downstream performance.

6. By strongly or perfectly aligning modalities, we risk making the two representations redundant, losing the complementary information that multimodality provides. We might also overfit to wrong correlations that are aligned but not causal. To design experiments to show that these risks are not present, we will measure alignment strength and perform ablation studies. The representation paper uses mutual nearest-neighbor as an alignment metric, which we can report. For the ablation study, we can compare performance of the fused model when we use only notes, only tabular, or both; if both consistently outperforms either alone, complementary information is still being used. We can also see by manual inspection, that through examining the notes and tabular features, it is evident that they are far from similar and holding the same information. 

#Part 2 (Homework Assignment), 100 points

**Reminder: there are 5 homework assignments, 7% each for 35% total of your final grade.**

For this assignment, we will finally begin playing with some of the concepts discussed in the class regarding multimodal modeling.

The first part will deal with Einsum and Tensors.

# Problem 1: Tensors (5 points)

(5 pts) Let's start with tensors. A tensor represents an N-th dimensional array of numbers. In machine learning, they are used to represent data as they can efficiently represent complex data to train with. We traditionally use PyTorch as the package of choice to work with tensors. Fill in the code below with the right tensor operations. Feel free to consult the documentation and the PyTorch tutorials for help.

In [11]:
import torch
mat_A = torch.rand(3, 2)
mat_B = torch.rand(2, 3)

In [ ]:
# Common PyTorch operations

# Adding
mat_C = torch.add(mat_A, mat_A)

# Transpose
mat_A_transpose = torch.transpose(mat_A, 0, 1)

# Matrix multiplication
mat_mult  = torch.matmul(mat_A, mat_B)

# Element-wise multiplication
mat_mult_elm = torch.mul(mat_A, mat_A)

# Create a tensor of size (4, 4) of ones
ones = torch.ones((4,4))

# Compute mean of A
mean_A = torch.mean(mat_A)

# Problem 2: Einsum (5 points)

(10 pts)
Now lets proceed with Einsum. This is a powerful, compact notation used for expressing complex tensor operations on multi-dimensional arrays using a simple string of index labels.

Here is a quick example of using einsum to multiply two matrices.

In [ ]:
A = torch.rand(3, 2)
B = torch.rand(2, 3)

C = torch.einsum('ij,jk->ik', A, B)
print(C)

tensor([[0.5922, 0.3843, 0.4396],
        [0.7406, 0.6216, 0.7325],
        [0.7258, 0.5192, 0.6012]])


The labels provide a shorthand as to what operation to do. Think of the left index as what is before, and the right as to what the dimensions of the final product should look like.

Now use this to do the other possible operations:

In [ ]:
a = torch.rand(3, 1)
b = torch.rand(3, 1)

A = torch.rand(3, 2)
B = torch.rand(2, 3)

# Dot Product of a and b
d_prod = torch.einsum('ij,ij->',a,b)

# Transpose using vector b
transpose = torch.einsum('ij->ji',b)

# Summation (element-wise and column-wise of A)
sum_element = torch.einsum('ij->',A)
sum_column = torch.einsum('ij->j',A)

# Diagonal of A
k = min(A.shape)
diag = torch.einsum('ii->i', A[:k, :k])

# Outer Product of a and b
outer = torch.einsum('ij,kl->ik',a,b)

In [ ]:
outer

tensor([[0.206950, 0.057152, 0.062243],
        [0.467900, 0.129216, 0.140727],
        [0.632071, 0.174554, 0.190103]])

In [ ]:
# Tests to verify that operations were done correctly
def to_list(t):
    """Convert tensor to Python list/scalar; pass through already Python types."""
    if isinstance(t, torch.Tensor):
        return t.detach().cpu().tolist()
    if isinstance(t, (int, float)):
        return t
    return t

def to_scalar(x):
    """Convert a 0-dim tensor or number to Python float for comparison."""
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu().item())
    return float(x)

def assert_close(a, b, rtol=1e-5, atol=1e-8):
    """Assert two scalars are equal within floating-point tolerance."""
    if isinstance(a, list) and isinstance(b, list):
        assert len(a) == len(b), f"List lengths differ: {len(a)} != {len(b)}"
        for i, (x, y) in enumerate(zip(a, b)):
            assert_close(x, y, rtol=rtol, atol=atol)
    else:
        a, b = float(a), float(b)
        assert abs(a - b) <= atol + rtol * abs(b), f"{a} != {b}"

def check_dot_product(ans, a, b):
    expected = sum(i * j for i, j in zip(to_list(a), to_list(b)))
    assert_close(to_scalar(ans), expected)

def check_transpose(ans, b):
    b_list = to_list(b)
    expected = [[row[i] for row in b_list] for i in range(len(b_list[0]))]
    assert_close(to_list(ans), expected)

def check_sum_element(ans, A):
    expected = sum(val for row in to_list(A) for val in row)
    assert_close(to_scalar(ans), expected)

def check_sum_column(ans, A):
    A_list = to_list(A)
    expected = [sum(row[i] for row in A_list) for i in range(len(A_list[0]))]
    assert_close(to_list(ans), expected)

def check_concat(ans, A, B):
    expected = to_list(A) + to_list(B)
    assert_close(to_list(ans), expected)

def check_diagonal(ans, A):
    A_list = to_list(A)
    expected = [A_list[i][i] for i in range(min(len(A_list), len(A_list[0])))]
    assert_close(to_list(ans), expected)

def check_outer_product(ans, a, b):
    a_l, b_l = to_list(a), to_list(b)
    expected = [[i * j for j in b_l] for i in a_l]
    assert_close(to_list(ans), expected)

In [ ]:
check_dot_product(d_prod, torch.flatten(a), torch.flatten(b))
check_transpose(transpose, b)
check_sum_element(sum_element, A)
check_sum_column(sum_column, A)
check_diagonal(diag, A)
check_outer_product(outer, torch.flatten(a), torch.flatten(b))

# Problem 3: Unimodal Models (10 points)

We now explore unimodal models and multimodal fusion. For the first part we will work on the image and audio digit dataset AV-MNIST to do digit classification. To benchmark effectiveness, we will use the [Multibench](https://arxiv.org/abs/2107.07502) benchmark. First, we will clone the repo, and get the necessary packages and dataset.

**Note: MAKE SURE YOU SWITCH TO A GPU TO RUN THE MODELS. RUNTIME -> CHANGE RUNTIME TYPE -> T4 GPU (or any other). Be mindful of Google's GPU limits based on what kind of account you own.**

**Also, if you are a student you should be able to have Colab Pro for free if you don't already. Take advantage of that!**

**THIS IS AN EXAMPLE, DO NOT BE RESTRICTED BY WHAT WE DO HERE WHEN YOU HAVE TO IMPLEMENT THIS FOR YOUR OWN DATASET.**

# Getting repo

In [4]:
!git clone https://github.com/pliang279/MultiBench.git
%cd MultiBench

/content/MultiBench


# Getting AV-MNIST dataset

In [3]:
!mkdir data
!pip install gdown
!pip install torch==2.3.1 torchvision==0.18.1 torchtext==0.18.0 torchaudio==2.3.1
!pip install memory_profiler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/1

In [8]:
!gdown 1KvKynJJca5tDtI5Mmp6CoRh9pQywH8Xp
!tar -xvzf avmnist.tar.gz

Downloading...
From (original): https://drive.google.com/uc?id=1KvKynJJca5tDtI5Mmp6CoRh9pQywH8Xp
From (redirected): https://drive.google.com/uc?id=1KvKynJJca5tDtI5Mmp6CoRh9pQywH8Xp&confirm=t&uuid=a12bc8f1-63d0-400e-b1b8-be53bd8654bf
To: /content/MultiBench/avmnist.tar.gz
100% 1.60G/1.60G [00:16<00:00, 98.9MB/s]
avmnist/
avmnist/test_labels.npy
avmnist/image/
avmnist/image/train_data.npy
avmnist/image/test_data.npy
avmnist/audio/
avmnist/audio/train_data.npy
avmnist/audio/test_data.npy
avmnist/train_labels.npy
avmnist/avmnist_temp/
avmnist/avmnist_temp/train_labels.npy
avmnist/avmnist_temp/image/
avmnist/avmnist_temp/image/test_data.npy
avmnist/avmnist_temp/image/train_data.npy
avmnist/avmnist_temp/test_labels.npy


In [9]:
# 1. Path to the folder you untarred
data_dir = '/content/MultiBench/avmnist'

from datasets.avmnist.get_data import get_dataloader
traindata, validdata, testdata  = get_dataloader(data_dir, batch_size=256)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


# Getting packages

In [5]:
import torch
import torch.nn as nn
import sys
import os
import torch.optim as optim
from tqdm import tqdm
from unimodals.common_models import GRU, MLP, Sequential, Identity
from training_structures.Supervised_Learning import train, test
import torch.nn.functional as F

We will now start by creating, training, and testing unimodal models for each of the AV-MNIST modalities.

# Audio

In [ ]:
class AudioModel(nn.Module):
    def __init__(self, input_dim=12544, hidden_dim=64, dropout_probability=0.1):
        super(AudioModel, self).__init__()
        self.conv = nn.Sequential(
            # Start with a stride of 2 to instantly cut data in half
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), # 1, 112, 112 -> 16, 56, 56
            nn.ReLU(),
            nn.MaxPool2d(2),                                     # 16, 56, 56 -> 16, 28, 28
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 16, 28, 28 -> 32, 14, 14
            nn.ReLU(),
            nn.Dropout2d(p=dropout_probability),
            nn.Flatten() # Only 6272 features now!
        )
        self.fc = nn.Linear(6272, 10)

    def forward(self, x):
        x = x.view(-1, 1, 112, 112)
        return self.fc(self.conv(x))

# Image

In [ ]:
class ImageModel(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(ImageModel, self).__init__()

        # input: [batch, 1, 28, 28]
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) # Reduces size by half

        # After two poolings: 28 -> 14 -> 7
        # Final flattened size: 64 channels * 7 * 7
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten all dimensions except batch
        x = torch.flatten(x, 1)
        return self.fc(x)

# Training and Testing

We use cross-entropy due to this being a classification task

In [26]:
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

# We use a scalar here to reduce system RAM use (to avoid crashing the session) while not impacting performance.
scaler = GradScaler()

def train_and_test_unimodal(model, train_loader, valid_loader, test_loader, modality_idx, epochs=10, lr=1e-3):
    device = torch.device("cuda")
    model.to(device)

    # Use CrossEntropyLoss for a classification task
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)

    best_valid_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            # batch[0] = images, batch[1] = audio
            x = batch[modality_idx].to(device).float()

            # Classification labels must be Long tensors, not Float
            y = batch[2].to(device).long().squeeze()

            optimizer.zero_grad()

            with autocast(device_type='cuda'):
                outputs = model(x)
                loss = criterion(outputs, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        # --- Validation Phase ---
        model.eval()
        valid_loss = 0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in valid_loader:
                x = batch[modality_idx].to(device).float()
                y = batch[2].to(device).long().squeeze()

                outputs = model(x)
                valid_loss += criterion(outputs, y).item()

                _, predicted = torch.max(outputs.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()

        avg_train = train_loss / len(train_loader)
        avg_valid = valid_loss / len(valid_loader)
        accuracy = 100 * correct / total

        if avg_valid < best_valid_loss:
            best_valid_loss = avg_valid
            torch.save(model.state_dict(), 'best_avmnist_model.pt')

        print(f"Epoch {epoch}: Train Loss: {avg_train:.4f} | Valid Acc: {accuracy:.2f}%")

    # Final Testing follows the same logic (CrossEntropy + Index 2)
    print("\n--- Final Evaluation Complete ---")
    model.load_state_dict(torch.load('best_avmnist_model.pt'))
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            x = batch[modality_idx].to(device).float()
            y = batch[2].to(device).long().squeeze()

            outputs = model(x)
            test_loss += criterion(outputs, y).item()

            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

        test_accuracy = 100 * correct / total
        test_loss /= len(test_loader)
        print(f"Final Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.2f}%")

# Training and testing for each modality:

# Audio:

In [34]:
audio_model = AudioModel()
train_and_test_unimodal(audio_model, traindata, validdata, testdata, modality_idx=1, epochs=10, lr=5e-3)

Epoch 0: Train Loss: 2.1300 | Valid Acc: 38.26%
Epoch 1: Train Loss: 2.0433 | Valid Acc: 39.46%
Epoch 2: Train Loss: 2.0277 | Valid Acc: 39.60%
Epoch 3: Train Loss: 2.0191 | Valid Acc: 40.38%
Epoch 4: Train Loss: 2.0124 | Valid Acc: 40.10%
Epoch 5: Train Loss: 2.0128 | Valid Acc: 40.58%
Epoch 6: Train Loss: 2.0090 | Valid Acc: 40.34%
Epoch 7: Train Loss: 2.0089 | Valid Acc: 40.44%
Epoch 8: Train Loss: 2.0042 | Valid Acc: 40.18%
Epoch 9: Train Loss: 2.0033 | Valid Acc: 41.16%

--- Final Evaluation Complete ---
Final Test Loss: 2.0358 | Test Accuracy: 38.99%


# Image:

In [31]:
image_model = ImageModel()
train_and_test_unimodal(image_model, traindata, validdata, testdata, modality_idx=0, epochs=30, lr=1e-4)

Epoch 0: Train Loss: 1.6271 | Valid Acc: 61.64%
Epoch 1: Train Loss: 1.1026 | Valid Acc: 63.26%
Epoch 2: Train Loss: 1.0259 | Valid Acc: 65.38%
Epoch 3: Train Loss: 0.9905 | Valid Acc: 67.08%
Epoch 4: Train Loss: 0.9683 | Valid Acc: 68.00%
Epoch 5: Train Loss: 0.9515 | Valid Acc: 67.52%
Epoch 6: Train Loss: 0.9415 | Valid Acc: 68.32%
Epoch 7: Train Loss: 0.9341 | Valid Acc: 68.72%
Epoch 8: Train Loss: 0.9274 | Valid Acc: 69.22%
Epoch 9: Train Loss: 0.9210 | Valid Acc: 68.74%
Epoch 10: Train Loss: 0.9159 | Valid Acc: 68.82%
Epoch 11: Train Loss: 0.9124 | Valid Acc: 68.76%
Epoch 12: Train Loss: 0.9100 | Valid Acc: 69.66%
Epoch 13: Train Loss: 0.9049 | Valid Acc: 69.52%
Epoch 14: Train Loss: 0.9052 | Valid Acc: 69.06%
Epoch 15: Train Loss: 0.9035 | Valid Acc: 69.46%
Epoch 16: Train Loss: 0.9012 | Valid Acc: 69.42%
Epoch 17: Train Loss: 0.8980 | Valid Acc: 69.92%
Epoch 18: Train Loss: 0.8956 | Valid Acc: 69.48%
Epoch 19: Train Loss: 0.8953 | Valid Acc: 69.30%
Epoch 20: Train Loss: 0.8931 |

Answer the following questions:

1. (5 points) Try to get the best performance out of each model by playing around with hyperparameters (hint: you may have to playing around and even add additional arguments to the layers like dropout, look at the documentation and look into how we can improve performace). List the best performance you were able to get and the hyperparameters you used.
2. (5 points) Compare the performances of each modality. What do these suggest to you? What could be done to get the worst performing ones to get closer to the best performing modality/model?


## Response:
1. With no hyperparameter, audio achieved test loss of 2.0123 and accuracy 41.15% and image achieved test loss of 0.9092 and accuracy 63.95%. After experimenting with learning rate, dropout layers, and epochs, I was able to improve these results. For images, I achieved test loss 0.8826 and accuracy 65.01% using dropout probability 0.2, learning rate 1e-4, and 25 epochs. For audio, hyperparameter search did not improve over the default setup. Using no dropout and learning rate of 1e-3 with 10 epochs (the original hyperparameters) led to the best accuracy and test loss.

2. Images clearly outperform audio on this digit-classification task: higher test accuracy and lower test loss. This indicates that, as a standalone modality, the image channel is more informative for AV-MNIST, which makes intuitive sense. To bring audio closer to the image modality, we can align before fusing: first learn a shared representation space so that audio and image features are aligned. Then, late or early fusion can combine these aligned representations, allowing the model to leverage both modalities and letting the stronger image modality provide signal to improve the audio encoder. 

# Problem 4: Multimodal Fusion (10 points)

Now you will play with multimodal fusion. Lets use a late fusion to improve our performance. We have provided some code with the hyperparameters to consider, but you are encouraged to play with them to try to get improvments. To make things simpler, the encoders for both modalities have been provided. However, some other parts are missing, so you will have to fill those in!

In [10]:
import torch.nn as nn
from unimodals.common_models import MLP
from fusions.common_fusions import MultiplicativeInteractions2Modal, Concat
from training_structures.Supervised_Learning import train

# Image Encoder
class ImageEncoder(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, output_dim)
        )
    def forward(self, x):
        return self.conv(x)

# Audio Encoder
class AudioEncoder(nn.Module):
    def __init__(self, output_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(), # 112 -> 56
            nn.MaxPool2d(2), # 56 -> 28
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(), # 28 -> 14
            nn.Flatten(),
            nn.Linear(64 * 14 * 14, output_dim)
        )
    def forward(self, x):
        return self.conv(x)

# TODO: Encoders - create the list of encoders, images should be first, then audio
encoders = [ImageEncoder(), AudioEncoder()]

# TODO: Use the concat fusion as the fusion of choice
fusion = Concat() # CODE HERE

# TODO: Create the head, which learns the joint features.
# This should be an MLP that takes with input size based
# on output size of your concationation, a hidden layer of size 256, and output layer
# of size 10.

head = MLP(128, 256, 10)# YOUR CODE

# Run Training
print("Starting Training...")
train(encoders, fusion, head, traindata, validdata, 5,
      task="classification", optimtype=torch.optim.AdamW, is_packed=False,
      lr=5e-4, save='avmnist_lmf.pt', weight_decay=0.001,
      objective=torch.nn.CrossEntropyLoss())

# Run Test
model = torch.load('avmnist_lmf.pt').cuda()
test(model, testdata, 'avmnist', is_packed=False, task="classification",
      criterion=torch.nn.CrossEntropyLoss(), no_robust=True)

Starting Training...
Epoch 0 train loss: tensor(0.8947, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 0 valid loss: tensor(0.7294, device='cuda:0') acc: 0.7258
Saving Best


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Epoch 1 train loss: tensor(0.7577, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 1 valid loss: tensor(0.7391, device='cuda:0') acc: 0.7256


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Epoch 2 train loss: tensor(0.7112, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 2 valid loss: tensor(0.7276, device='cuda:0') acc: 0.7314
Saving Best


...

Epoch 3 train loss: tensor(0.6789, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 3 valid loss: tensor(0.7243, device='cuda:0') acc: 0.7348
Saving Best


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Epoch 4 train loss: tensor(0.6468, device='cuda:0', grad_fn=<DivBackward0>)
Epoch 4 valid loss: tensor(0.7372, device='cuda:0') acc: 0.725
Training Time: 75.08934473991394
Training Peak Mem: 9765.87109375
Training Params: 1077258


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


acc: 0.6972
Inference Time: 2.985517978668213
Inference Params: 1077258


Answer the following:
1. (2 points) Sometimes when training you may notice the model gets stuck in a range of loss and never seems to get it's loss down. What does this suggest? What are some ways you can fix this?
2. (2 points) What are some other fusion methods we could use that we could use? Would they lead to improvements compared to early fusion?
3. (6 points) Explain the difference between early fusion techniques and late fusion techniques. Be sure to discuss their benefits and tradeoffs.

## Response:
1. When this happens, it usually indicates that the model is stuck in a local minimum or a plateau in the loss, which prevents further learning. This could be due to using an inappropriate learning rate, poor weight initialization, or unsuitable optimizer choice. One way to fix this is by increasing the learning rate, switching to a different optimizer (such as Adam), changing the weight initialization, or using learning rate scheduling or regularization techniques to help the model escape shallow minima or plateaus.
2. Other fusion methods include tensor fusion, low-rank tensor fusion, gated fusion, and multiplicative/interactions-based fusion. These approaches can capture more complex cross-modal relationships. While early fusion is often effective, in some cases methods like gated or tensor fusion can outperform it by leveraging richer interactions.
3. Early fusion techniques merge multimodal data at the raw feature level or early in the model, allowing the model to learn joint feature representations from combined input modalities. This makes it possible to capture complex dependencies and interactions between modalities. However, early fusion often requires the modalities to be temporally or spatially aligned, and can be sensitive to missing or noisy data from any modality. In contrast, late fusion techniques process each modality independently through separate models or encoders, and then combine their outputs or decisions at a later stage. The main advantages of late fusion are its flexibility, robustness to missing modalities, and the ability to utilize specialized or pre-trained models for each modality. However, late fusion cannot capture as rich interactions between modalities as early fusion can, since the information is only combined at the decision or feature level rather than throughout the whole model.

# Problem 5: Other Fusion Techniques (30 points)

Now, we want you to try implementing some of these fusion techniques on your dataset! For this part, you will implement these fusion techniques:
1. Early fusion
2. Late fusion
3. TensorFusion
4. Low-Rank Tensor (LMF) Fusion


**You cannot just import and use the functions available in Multibench to do this. In addition, use einsum where applicable. TO RECIEVE FULL CREDIT, THE FUSIONS YOU IMPLEMENTATION MUST WORK WITH THE DATASET YOU CREATED FROM HOMEWORK 1. YOU WILL HAVE TO CREATE A SIMPLE MODEL FOR EACH FUSION TECHNIQUE TO PLAY WITH.**

**(5 points)** In your write up, report the best validation accuracies of your multimodal model (don't forget to include what hyperparameters you included) after training and any modifications that had to be done to your data or model to be able to train on it. In addition, talk about which technique you believe would be best for your dataset and why that is.

**Design the fusion classes with the modalities you are specifically working with in mind. The example we worked through above with MOSI was meant as a showcase of fusion in action - we do not require you to use text, video and audio as the modalities. Use whichever ones you are working with!**

The code below provided is to be filled in with the models you set up for each technique. For an example, the first fusion technique has been done for you.

**Answer Here:**

In [1]:
# Imports incase you need them again! Feel free to include anything else you need
import torch
from torch import nn
from torch.nn import functional as F
import pdb
from torch.autograd import Variable

# Early Fusion

In [ ]:
class EarlyFusion(nn.Module):
  def __init__(self):
    super(EarlyFusion, self).__init__()

  def forward(self, x):
    return torch.einsum('bi,bj->bij', x[0], x[1])

# (5 Points) Late Fusion

In [ ]:
class LateFusion(nn.Module):
  def __init__(self):
    pass

  def forward(self, x):
    pass

NameError: name 'nn' is not defined

# (5 points) Tensor Fusion

In [ ]:
class TensorFusion(nn.Module):
  def __init__(self):
    pass

  def forward(self, x):
    pass

# (5 Points) Low-Rank Tensor Fusion (LMF) Fusion

In [ ]:
class LMFFusion(nn.Module):
  def __init__(self):
    pass

  def forward(self, x):
    pass

(10 points) In addition, create some visualizations of the following for each fusion:

* Number of parameters for each model (unimodal and multimodal)
* Memory Use
* Time until convergence

You are free to plot them here or through other means (like wandb). After doing so, discuss what are the pros and cons of unimodal versus multimodal models.

# Problem 6: Contrastive Learning (30 points)

For the next part of this HW, we will focus on contrastive learning. As a reminder, contrastive learning is a local, discrete alignment method used in machine learning. To explore this, we look at [CLIP](https://arxiv.org/pdf/2103.00020), a multimodal model developed by OpenAI that uses contrastive learning to align visual and textual data together.

**THIS IS JUST AN EXAMPLE, DO NOT LET THIS RESTRICT THE IMPLEMENTATION YOU WILL BE DOING.**

In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-8a206fek
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-8a206fek
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=ca8a5f278ed012078baeed3a2a84a3db5796719ed214ab98c5fae7bc4d192588
  Stored in directory: /tmp/pip-ephem-wheel-cache-ju96rty9/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
# Packages to import
import transformers
import torch
import clip
from PIL import Image
import requests
from io import BytesIO

First, we create the model.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP on {device}...")
model, preprocess = clip.load("ViT-B/32", device=device)

Loading CLIP on cuda...


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 161MiB/s]


Next, we will load an image to use. Note that we cannot use the MOSI dataset - we need to use raw data and the data points from the dataset already have extracted features. Upload a picture of someone smiling to use for this example (you can just find one online, save it and add to here).

In [ ]:
image_filename = "../smiling_person.png" # REPLACE WITH YOUR FILE
image = Image.open(image_filename).convert("RGB")

Now, we will prepare the prompt to use.

In [ ]:
# Options to pick from
text_options = ["a photo of a sad person", "a photo of a happy person", "a photo of an angry person"]
image_input = preprocess(image).unsqueeze(0).to(device)
text_inputs = clip.tokenize(text_options).to(device)

Now, let's run the inference and get the results!

In [ ]:
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_inputs)

    # Normalize features
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Calculate similarity (Dot Product)
    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
    values, indices = similarity[0].topk(3)

print(f"\nImage classified against: {text_options}")
print("-" * 30)
for value, index in zip(values, indices):
    print(f"{text_options[index]:>30s}: {100 * value.item():.2f}%")


Image classified against: ['a photo of a sad person', 'a photo of a happy person', 'a photo of an angry person']
------------------------------
     a photo of a happy person: 98.44%
    a photo of an angry person: 1.09%
       a photo of a sad person: 0.49%


(10 pts) We will now create, train and run zero-shot classification using contrastive learning for your own dataset. Fill in the missing information below for a generalize contrastive learning model. The training and zero-shot classification functions have been provided to you, through you may need to make slight modifications based on your dataset setup. **Design the model keeping in mind the modalities that you are specifically using. THE CLIP EXAMPLE ABOVE IS JUST TO SHOW CONTRASTIVE LEARNING IN ACTION - WE ARE NOT REQUIRING THAT YOU USE TEXT AND IMAGE AS THE MODALITIES OF CHOICE.** Try various queries, projectors, and settings on your dataset!

**You must use einsum where applicable.**


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# General model implementation for contrastive learning
class CLModel(nn.Module):
  def __init__(self, dim_x1, dim_x2, embedded_dim, temp):
    # TODO:
    # 1. Create Encoders for modalities
    # 2. Create a projector, which maps specific modality dimensions to a shared space.
    #     do this for each modality. (hint: fusions!)
    # 3. Create learnable temperature (this has already been done for you)

    # 1.

    # 2.

    # 3.
    self.scale = nn.Parameter(torch.ones([]) * np.log(1/temp))

  def forward(self, x1, x2):
    # TODO:
    # 1. Extract the raw features
    # 2. Project them to the embedding space
    # 3. Normalize vectors and return

    # YOUR CODE HERE
    pass

# Contrastive loss. This pulls positives together and pulls negatives apart
class ContrastiveLoss(nn.module):
  def __init__(self, model):
    # TODO: Initialize model and loss function as cross entropy loss
    self.model = # YOUR CODE
    self.loss_fn = # YOUR CODE

  def forward(self, x1_emb, x2_emb):
    # TODO:
    # 1. Get the batch size (hint: you can get this
    #    from the dimensions of your embedded space)
    # 2. Create similarity matrix using einsum
    # 3. Create labels (hint: the coorect match for index i is label i)
    # 4. Compute Symmetric loss (loss amongst rows + loss amongst columns)/2
    pass





SyntaxError: incomplete input (ipython-input-3452034263.py, line 26)

In [ ]:
import torch.optim as optim
# Training function
def train_model(model, contrastive_loss, dataloader, num_epochs=5, learning_rate=3e-4, device='cpu'):

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.to(device)
    model.train()
    print(f"Starting training for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        epoch_loss = 0.0

        for batch_idx, (data_a, data_b) in enumerate(dataloader):
            data_a, data_b = data_a.to(device), data_b.to(device)

            optimizer.zero_grad()

            emb_a, emb_b = model(data_a, data_b)

            loss = contrastive_loss(emb_a, emb_b, model.logit_scale)

            loss.backward()

            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_loss:.4f}")

In [ ]:
# After training, we can now do zero-shot prediction
@torch.no_grad()
def predict_best_match(model, query_input, candidate_inputs, device):
    model.eval()

    query_feat = model.encoder_a(query_input.unsqueeze(0).to(device))
    query_emb = F.normalize(model.projector_a(query_feat), dim=1)

    cand_feat = model.encoder_b(candidate_inputs.to(device))
    cand_emb = F.normalize(model.projector_b(cand_feat), dim=1)

    scores = torch.einsum('id, jd -> ij', query_emb, cand_emb)

    best_match_idx = scores.argmax().item()

    print(f"Best match: {best_match_idx} with score {scores[0, best_match_idx].item()}")

    return best_match_idx, scores

Now answer some of these questions:

1. (5 points) Any suprising results from using this on your dataset?
2. (5 points) Typically, cross-entropy loss is used in this contrastive learning, why is this the case?
3. (10 points) Create some visual examples of the data post alignment. Can you point out samples where the alignment worked and where it failed? Why do you suspect that is?

# Problem 7: Reflection (10 points)

Now we'll take some time to reflect on this homework. Take some time to discuss the following:

1. (5 points) What concept did you find the most interesting?
2. (5 points) Which concepts (if any) do you see being useful towards your goal? Why? If there was none, discuss why.
3. (0 points, optional) Is there a topic that was discussed during lectures up to the release of the assignment that you wished was covered in the homework? Any from the assignment that you wanted there to be touched upon more? Any feedback you have in general for homeworks or the class?